# SNCP-PPO Social Navigation — Colab Notebook

End-to-end notebook for training and evaluating an **LTC + PPO** crowd-aware navigation policy on Google Colab.

## What you'll do

1. **Setup** — clone repo, install deps, mount Drive (optional)
2. **Smoke test** — verify the environment + model + training loop
3. **Train** — full curriculum-based training with multi-scenario holdout
4. **Evaluate** — compare new checkpoint against existing v2 baseline
5. **Visualize** — trajectory plots + GIFs across all scenarios
6. **Analyze** — read training CSV, plot learning curves

## Architecture recap

- **Policy**: SNCPPolicy (3 LTC cells: temporal/spatial/node + attention + actor-critic heads)
- **Algorithm**: PPO with clipped value loss, GAE with truncation bootstrap, BPTT over LTC subsequences
- **Curriculum**: 5 phases, N=1→2→3→4→5 pedestrians, vpref 0.15→0.50
- **Reward**: goal-approach + collision + per-human-normalized comfort (I_sp / N)
- **Best-checkpoint metric**: `min(success across holdout scenarios)` — rewards generalists

## Colab tips

- Use **Runtime → Change runtime type → T4 GPU** (free tier)
- Free tier has ~12 hour session limit — full 1500-ep training fits comfortably (~5h)
- Mount Drive (Section 1.5) to persist checkpoints/logs across sessions

## 1. Setup

### 1.1 Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone repository

Replace `heimdilon` with your GitHub user if you forked the repo.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!ls -la

### 1.3 Install dependencies

PyTorch comes pre-installed on Colab; we just add `ncps`, `gymnasium`, and confirm versions.

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch    {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'         device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')
print(f'matplotlib {matplotlib.__version__}')

### 1.4 (Optional) Mount Google Drive

If you want checkpoints/logs to persist across Colab sessions, mount Drive and we'll symlink `checkpoints/` and `logs/` into a Drive folder.

**Skip this cell if you're just running a quick experiment.**

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    # Replace local dirs with symlinks to Drive
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            # Backup existing local dir then symlink
            import shutil
            for f in os.listdir(local):
                src = f'{local}/{f}'
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files will be lost when Colab session ends.')

## 2. Smoke tests

Three fast self-tests:
1. Environment reset/step + observation shapes
2. Model forward pass
3. 50-episode mini-training (verifies the full pipeline + new Path A changes)

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode smoke training — verifies curriculum, holdout, value clipping, LR schedule.
# Should complete in ~5 minutes on a T4 GPU.
!python -m sncp_ppo.train \
    --episodes 50 \
    --num_humans 5 \
    --seed 42 \
    --eval_freq 25 \
    --holdout_episodes 3 \
    --holdout_scenarios easy hard \
    --update_freq 5 \
    --log_freq 10 \
    --save_path checkpoints/sncp_ppo_smoke.pt

## 3. Full training

The canonical run is **1500 episodes** (~5h on a T4 GPU, including holdouts). For shorter Colab sessions or quicker iteration, drop to 750 or 500 episodes — the curriculum percentages auto-scale.

### Key arguments

| Argument | Meaning | Tuning notes |
|---|---|---|
| `--episodes` | Total training episodes | 1500 = canonical, 500 = quick |
| `--lr` / `--lr_end_factor` | Base lr with linear decay | 1e-4 → 1e-5 default |
| `--holdout_scenarios easy hard` | Evaluate generalist metric | Add `medium` for stricter |
| `--holdout_episodes 30` | Episodes per holdout per scenario | ≥20 for stable estimates |
| `--eval_freq 50` | Episodes between holdouts | Lower = finer eval curve |
| `--curriculum_*_until` | Override 5-phase boundaries | Defaults: 10/25/50/75% |

In [ ]:
# Customize before running
EPISODES = 1500       # 500 for quick iteration, 1500 for canonical
SEED = 42
SAVE_PATH = 'checkpoints/sncp_ppo_v3_colab.pt'

import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', str(EPISODES),
    '--num_humans', '5',
    '--seed', str(SEED),
    '--lr', '1e-4',
    '--lr_end_factor', '0.1',
    '--holdout_scenarios', 'easy', 'hard',
    '--holdout_episodes', '30',
    '--eval_freq', '50',
    '--update_freq', '5',
    '--log_freq', '20',
    '--save_path', SAVE_PATH,
]
print('Running:', ' '.join(cmd))
print('=' * 80)
# Stream output line by line so we see progress live
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

### Resuming a previous Colab session

If Colab disconnected mid-training: with `USE_DRIVE=True`, your latest periodic checkpoint (`sncp_ppo_v3_colab_ep<N>.pt`) is still in Drive. There's no built-in resume CLI — easiest path is to just rerun from scratch with a different `--seed` (each seed gives a fresh trajectory). For exact resume support, see the **Roadmap** at the end of this notebook.

## 4. Evaluation

Run 100 deterministic episodes per scenario to characterize the trained policy.

In [ ]:
CHECKPOINT = 'checkpoints/sncp_ppo_v3_colab.pt'  # or any other .pt in checkpoints/
EVAL_SEED = 100  # different from training seeds for fair eval
EVAL_EPISODES = 100

for scenario in ['easy', 'easy_plus', 'medium', 'hard', 'extreme']:
    print(f'\n{"=" * 80}\nScenario: {scenario}\n{"=" * 80}')
    # num_humans by scenario (mirrors holdout config)
    n_humans = {'easy': 1, 'easy_plus': 2, 'medium': 3, 'hard': 5, 'extreme': 5}[scenario]
    !python test_eval.py \
        --checkpoint {CHECKPOINT} \
        --num_humans {n_humans} \
        --scenario {scenario} \
        --n_episodes {EVAL_EPISODES} \
        --seed {EVAL_SEED} 2>&1 | tail -10

### Compare with shipped v2 baseline (if checkpoints included in repo)

In [ ]:
import os

if os.path.exists('checkpoints/sncp_ppo_v2.pt'):
    print('Evaluating v2 baseline (pre-Path-A, has known catastrophic-forgetting issues)')
    for scenario, n in [('easy', 1), ('hard', 5)]:
        print(f'\n--- v2 on {scenario}/{n}h ---')
        !python test_eval.py --checkpoint checkpoints/sncp_ppo_v2.pt \
            --num_humans {n} --scenario {scenario} --n_episodes 50 --seed 100 2>&1 | tail -8
else:
    print('v2 checkpoint not in repo. Train and save one, or clone the full repo with checkpoints.')

## 5. Visualize trajectories

Generate trajectory plots (PNG) and animated GIFs to inspect what the policy is doing visually.

In [ ]:
# Single trajectory plot — finds first successful episode out of 20 tries and plots it
!python visualize_trajectory.py \
    --checkpoint {CHECKPOINT} \
    --output trajectory_plot.png \
    --num_humans 5 \
    --scenario hard \
    --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))

In [ ]:
# Animated GIF for a single scenario
!python visualize_trajectory_gif.py --checkpoint {CHECKPOINT}

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))

In [ ]:
# All scenarios as separate GIFs (easy/medium/hard/extreme)
!python visualize_all_scenarios_gif.py --checkpoint {CHECKPOINT}

from IPython.display import Image, display
for sc in ['easy', 'medium', 'hard', 'extreme']:
    path = f'{sc}_trajectory.gif'
    if os.path.exists(path):
        print(f'\n--- {sc} ---')
        display(Image(path))

## 6. Training curves analysis

Plot the learning trajectory with per-scenario holdout lines and the generalist `min(success)` dashed line that drove best-checkpoint selection.

In [ ]:
import glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if not csv_files:
    print('No training CSVs found. Run section 3 first.')
else:
    latest_csv = csv_files[-1]
    print(f'Plotting: {latest_csv}')
    !python plot_training.py --csv {latest_csv} --output training_curves_colab.png --window 50
    from IPython.display import Image, display
    display(Image('training_curves_colab.png'))

### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout summary (per-scenario success rate at each eval):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        # Take rows where holdout changed (event points only)
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

If you used Drive (Section 1.4), checkpoints + logs already there. Otherwise download key artifacts before the session ends.

In [ ]:
from google.colab import files
import glob

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    # Best checkpoint
    if os.path.exists(SAVE_PATH):
        files.download(SAVE_PATH)
    # Latest training CSV + plot
    for pattern in ['logs/training_*.csv', 'training_curves_colab.png']:
        for f in sorted(glob.glob(pattern))[-1:]:
            files.download(f)

## 8. Notes & roadmap

### Known limitations of v3 (Path A complete)

- Observation lacks **human velocity** — policy must infer it via LTC dynamics, which is slow. Adding `(vx, vy)` to `spatial_edges` is in **Yol B**.
- No exact training resume — only periodic checkpoint snapshots.
- BPTT subsequences (`seq_len=16`) may be too short for LTC time constants to fully shape — try 32 if instability appears.

### What Path A changed vs v2

| | v2 (pre-Path A) | v3 (Path A) |
|---|---|---|
| Comfort reward | `-0.5 * I_sp` (scales with N) | `-0.5 * I_sp / N` (phase-invariant) |
| Curriculum | 4 phases (1→3→5 humans) | **5 phases** (1→2→3→4→5) |
| Value loss | Unbounded MSE | **Clipped** (OpenAI standard) |
| Learning rate | Fixed `3e-4` | **LinearLR** `1e-4 → 1e-5` |
| Holdout | Single scenario, lucky-seed false positives | **Multi-scenario**, `min(success)` metric |
| Holdout config | Used current curriculum N | **Canonical N per scenario** (hard=5, easy=1) |

### Going further

1. **Hyperparameter sweep** — try `--seed 1,2,3,4,5` to bound noise of the result.
2. **Mixed-difficulty replay** — re-include older-phase episodes (needs spatial_edges padding to max_humans).
3. **Architecture ablation** — swap one or more LTC cells with GRU to isolate LTC's contribution (Yol B).
4. **Real-robot sim2real** — `waffle_ros/` has a ROS node skeleton; calibrate noise and dynamics gap.

### Repo

- Source: https://github.com/heimdilon/sncp-ppo-crowdnav
- Issues / PRs welcome.